# Lesson 9.3 - Advanced NLP & LLM Applications (seq2seq/LLM demo)

## Objectives

- Run seq2seq summarization with a small transformer model.
- Demonstrate structured output prompting.
- Connect outputs to evaluation and safety concerns.


In [ ]:
from __future__ import annotations

from transformers import pipeline
import json


## Summarization with Seq2Seq Model

This demo uses a compact summarization model to keep runtime manageable.


In [ ]:
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

text = (
    "Reinforcement learning is a machine learning paradigm where an agent learns to make decisions "
    "by interacting with an environment. The agent receives rewards and aims to maximize cumulative "
    "return over time. Practical RL often struggles with sample inefficiency and training instability, "
    "which motivates replay buffers, target networks, and robust evaluation protocols."
)

summary = summarizer(text, max_length=60, min_length=20, do_sample=False)[0]["summary_text"]
print(summary)


## Translation-Style Prompting with Text2Text


In [ ]:
text2text = pipeline("text2text-generation", model="google-t5/t5-small")

prompt = "translate English to German: Deep learning systems require careful evaluation and monitoring."
translated = text2text(prompt, max_new_tokens=64)[0]["generated_text"]
print(translated)


## Structured Output Prompting (JSON Template)


In [ ]:
analysis_prompt = (
    "Summarize this issue and output JSON with keys: risk, mitigation, owner. "
    "Issue: The chatbot hallucinates unsupported legal clauses in contract summaries."
)

structured = text2text(analysis_prompt, max_new_tokens=128)[0]["generated_text"]
print(structured)

# Optional post-processing skeleton:
def try_parse_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        return {"raw": s}

try_parse_json(structured)


## Connect to Theory

- Seq2seq summarization uses conditional generation from source text.
- Text2text models unify multiple tasks under prompt conditioning.
- Structured outputs increase integration reliability for downstream systems.


## Business Case Studies & Exceptions

### Case A: Enterprise Support Summarization

Teams use summarization to compress long ticket threads. Strong retrieval grounding and human QA are needed for policy-sensitive responses.

### Case B: Developer Copilot Workflows

Code assistants reduce boilerplate and speed reviews, but need security/static checks to prevent vulnerable suggestions.

### Exceptions

- For strict deterministic transformations, rule-based pipelines may beat LLMs.
- In small, well-labeled domains, compact supervised NLP models may be cheaper and more controllable.


## Interview Questions & Answers

1. **What is seq2seq modeling?**  
   Mapping one sequence to another using encoder-decoder architectures.
2. **What is masked LM vs causal LM?**  
   Masked predicts hidden tokens; causal predicts next token autoregressively.
3. **Why use instruction tuning?**  
   To improve general instruction-following behavior.
4. **What is RLHF used for?**  
   Aligning model outputs with human preferences and safety goals.
5. **How do you evaluate summarization quality?**  
   Automatic metrics + human factuality/utility checks.
6. **Why is groundedness important?**  
   It reduces hallucinated claims.
7. **What is structured output?**  
   Constraining generations to typed schemas like JSON.
8. **How do you mitigate jailbreak risk?**  
   Prompt policies, filtering, routing, and adversarial tests.
9. **When use smaller models in production?**  
   When latency/cost/privacy constraints are strict.
10. **How would you productionize this demo?**  
   Add retrieval, eval harness, monitoring, and fallback logic.
